# Interpretability Analysis

Dedicated evidence notebook for model interpretation. It reads artifacts generated by the linear and tree pipelines and does not retrain models or recompute fitted estimators.

## Interpretability Scope

This notebook treats coefficients, impurity importance, permutation importance, SHAP, and partial dependence as fitted-model diagnostics. They support cautious interpretation of predictive behavior, not causal mechanism claims. The target remains environmentally related patenting share, and all interpretations should be read against the persistence baseline and latest-period test gap documented in the modeling notebooks.

In [ ]:
# ----- Project setup -----
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import Image, Markdown, display


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "3_models" / "scripts" / "model_config.py").exists():
            return candidate
    raise FileNotFoundError(f"Could not locate project root from {start}.")


ROOT = _find_project_root(Path(os.getcwd()).resolve())
SCRIPTS = ROOT / "3_models" / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from model_config import (  # noqa: E402
    COEFFICIENTS_OUTPUT,
    ERROR_BY_TARGET_QUANTILE_OUTPUT,
    ERROR_BY_YEAR_OUTPUT,
    FIGURE_INDEX_OUTPUT,
    FIGURES_DIR,
    OUTPUT_DIR,
    TARGET_CORRELATIONS_OUTPUT,
    TOP_ERRORS_OUTPUT,
    TREE_FIGURE_INDEX_OUTPUT,
    TREE_FIGURES_DIR,
    TREE_HISTORICAL_DELTA_OUTPUT,
    TREE_IMPORTANCE_OUTPUT,
    TREE_OUTPUT_DIR,
    TREE_PARTIAL_DEPENDENCE_OUTPUT,
)

PAPER_FIGURES_DIR = ROOT / "4_analysis" / "figures" / "paper"
PERMUTATION_IMPORTANCE_OUTPUT = TREE_OUTPUT_DIR / "tree_model_permutation_importance.csv"
PERMUTATION_IMPORTANCE_FIGURE = PAPER_FIGURES_DIR / "fig7_permutation_importance.png"
PERSISTENCE_ADJUSTED_COMPARISON_OUTPUT = OUTPUT_DIR / "persistence_adjusted_model_family_comparison.csv"
PERSISTENCE_ADJUSTED_SUMMARY_OUTPUT = OUTPUT_DIR / "persistence_adjusted_model_family_summary.csv"
PERSISTENCE_ADJUSTED_NATIVE_IMPORTANCE_OUTPUT = OUTPUT_DIR / "persistence_adjusted_native_importance.csv"
PERSISTENCE_ADJUSTED_TOP_FEATURES_OUTPUT = OUTPUT_DIR / "persistence_adjusted_top_interpretation_features.csv"
PERSISTENCE_LEVEL_CORRECTION_OUTPUT = OUTPUT_DIR / "persistence_adjusted_level_correction_predictions.csv"
PERSISTENCE_LEVEL_CORRECTION_SUMMARY_OUTPUT = OUTPUT_DIR / "persistence_adjusted_level_correction_summary.csv"
PERSISTENCE_ADJUSTED_RUN_SUMMARY_OUTPUT = OUTPUT_DIR / "persistence_adjusted_run_summary.md"
PERSISTENCE_ADJUSTED_FAMILY_FIGURE = PAPER_FIGURES_DIR / "fig12_persistence_adjusted_family_comparison.png"
PERSISTENCE_ADJUSTED_FEATURE_FIGURE = PAPER_FIGURES_DIR / "fig13_persistence_adjusted_delta_feature_roles.png"


def project_path(path_value) -> Path:
    path = Path(path_value)
    return path if path.is_absolute() else ROOT / path


def read_csv_or_empty(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def show_png(path: Path) -> None:
    path = project_path(path)
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"Missing figure: {path}")


def show_indexed_figure(index_path: Path, figure: str, fallback: Path) -> None:
    index = read_csv_or_empty(index_path)
    if not index.empty and {"figure", "path"}.issubset(index.columns):
        matches = index[index["figure"] == figure]
        if not matches.empty:
            show_png(project_path(matches.iloc[0]["path"]))
            return
    show_png(fallback)


print(f"Project root: {ROOT}")


## Artifact Availability

The report is intentionally artifact-first. If a row is missing, regenerate the corresponding modeling pipeline before interpreting the section.

In [ ]:
artifacts = [
    ("linear_coefficients", COEFFICIENTS_OUTPUT),
    ("linear_figure_index", FIGURE_INDEX_OUTPUT),
    ("tree_historical_baseline_delta", TREE_HISTORICAL_DELTA_OUTPUT),
    ("tree_importance", TREE_IMPORTANCE_OUTPUT),
    ("tree_figure_index", TREE_FIGURE_INDEX_OUTPUT),
    ("tree_partial_dependence", TREE_PARTIAL_DEPENDENCE_OUTPUT),
    ("tree_permutation_importance", PERMUTATION_IMPORTANCE_OUTPUT),
    ("persistence_adjusted_family_comparison", PERSISTENCE_ADJUSTED_COMPARISON_OUTPUT),
    ("persistence_adjusted_family_summary", PERSISTENCE_ADJUSTED_SUMMARY_OUTPUT),
    ("persistence_adjusted_native_importance", PERSISTENCE_ADJUSTED_NATIVE_IMPORTANCE_OUTPUT),
    ("persistence_adjusted_top_features", PERSISTENCE_ADJUSTED_TOP_FEATURES_OUTPUT),
    ("persistence_level_correction_predictions", PERSISTENCE_LEVEL_CORRECTION_OUTPUT),
    ("persistence_level_correction_summary", PERSISTENCE_LEVEL_CORRECTION_SUMMARY_OUTPUT),
    ("persistence_adjusted_run_summary", PERSISTENCE_ADJUSTED_RUN_SUMMARY_OUTPUT),
    ("linear_top_errors", TOP_ERRORS_OUTPUT),
    ("linear_error_by_year", ERROR_BY_YEAR_OUTPUT),
    ("linear_error_by_target_quantile", ERROR_BY_TARGET_QUANTILE_OUTPUT),
]
display(pd.DataFrame([
    {"artifact": name, "path": str(path.relative_to(ROOT) if path.exists() else path), "exists": path.exists()}
    for name, path in artifacts
]))


## Performance Boundary

Interpretability is only useful inside the model's demonstrated predictive boundary. The tree model explanations below are bounded by the historical-persistence comparison; if the selected model trails the country last-observed baseline, explanatory claims should remain descriptive and model-internal.

In [ ]:
historical_delta = read_csv_or_empty(TREE_HISTORICAL_DELTA_OUTPUT)
if not historical_delta.empty:
    keep = [
        "selected_model",
        "baseline_model",
        "selected_mae",
        "baseline_mae",
        "delta_mae_selected_minus_baseline",
        "selected_beats_baseline",
        "professor_interpretation",
    ]
    display(historical_delta.loc[:, [column for column in keep if column in historical_delta]].round(4))


## Global Importance

Linear coefficients and tree impurity importance give two complementary views. Coefficients are easiest to inspect for direction in the selected linear model, while tree importance summarizes split-driven predictive contribution and can be biased toward high-variance continuous predictors.

In [ ]:
show_indexed_figure(FIGURE_INDEX_OUTPUT, "coefficients", FIGURES_DIR / "linear_model_coefficients.png")
coefficients = read_csv_or_empty(COEFFICIENTS_OUTPUT)
if not coefficients.empty:
    # Filter active predictors with abs_coefficient > 1e-12.
    nonzero_coefficients = coefficients[coefficients["abs_coefficient"] > 1e-12].copy()
    if nonzero_coefficients.empty:
        print("Selected linear model has no nonzero coefficients above tolerance.")
    else:
        display(
            nonzero_coefficients.sort_values("abs_coefficient", ascending=False)
            .loc[:, ["feature", "coefficient", "abs_coefficient"]]
            .head(10)
            .round(4)
        )

show_indexed_figure(TREE_FIGURE_INDEX_OUTPUT, "importance", TREE_FIGURES_DIR / "tree_model_feature_importance.png")
tree_importance = read_csv_or_empty(TREE_IMPORTANCE_OUTPUT)
if not tree_importance.empty:
    display(tree_importance.loc[:, ["feature", "importance"]].head(10).round(4))


## SHAP Summary

SHAP is used here as a tree-model diagnostic for relative contribution patterns. It should be interpreted as model explanation, not as proof that a predictor causally changes environmental patenting.

In [ ]:
show_indexed_figure(
    TREE_FIGURE_INDEX_OUTPUT,
    "shap_summary",
    TREE_FIGURES_DIR / "tree_model_shap_summary.png",
)
print("Artifact key: tree_model_shap_summary")


## Permutation Importance

Permutation importance is the robustness counterpart to impurity importance. It is used here as a post-hoc test-block diagnostic, not for model selection or variable discovery. It asks how much predictive performance deteriorates when a feature is shuffled, so it is useful for checking whether the split-based ranking is overly model-internal.

In [ ]:
show_png(PERMUTATION_IMPORTANCE_FIGURE)
print("Artifact key: fig7_permutation_importance")
permutation_importance = read_csv_or_empty(PERMUTATION_IMPORTANCE_OUTPUT)
if not permutation_importance.empty:
    display(permutation_importance.head(10).round(4))


## Partial Dependence

Partial dependence shows the fitted tree model's average response across train+validation reference rows for top impurity-ranked predictors. It is a response diagnostic only. In correlated country-panel data, PDP can average over feature combinations that are sparse or absent in the observed data.

In [ ]:
show_indexed_figure(
    TREE_FIGURE_INDEX_OUTPUT,
    "partial_dependence",
    TREE_FIGURES_DIR / "tree_model_partial_dependence.png",
)
partial_dependence = read_csv_or_empty(TREE_PARTIAL_DEPENDENCE_OUTPUT)
if not partial_dependence.empty:
    display(partial_dependence.round(4).head(15))


## Persistence-Adjusted Target Diagnostics

The `delta_lag1` task asks whether the predictors explain annual within-country movement after removing the previous-year level. This is the main supplemental target. The `log_ratio_lag1` task is treated as robustness, because it changes the target scale and drops nonpositive pairs where needed. These results answer a different question from the level-target notebook: not which countries have structurally high environmental patent share, but which predictors help anticipate changes around each country's recent baseline.

## Delta Target Performance

The correct baseline for the adjusted target is the zero-change baseline. Negative `delta_test_mae_vs_zero_change` means lower MAE than predicting no annual movement. The table below is the selected best-family summary for `delta_lag1`; the figure and the comparison slice show the full family comparison. Use the signed delta as the primary evidence; `beats_zero_change_baseline` is descriptive only, and near-zero differences should not be treated as material. Validation-to-test degradation remains a boundary on interpretation.

In [ ]:
show_png(PERSISTENCE_ADJUSTED_FAMILY_FIGURE)
print("Artifact key: fig12_persistence_adjusted_family_comparison")
persistence_summary = read_csv_or_empty(PERSISTENCE_ADJUSTED_SUMMARY_OUTPUT)
persistence_comparison = read_csv_or_empty(PERSISTENCE_ADJUSTED_COMPARISON_OUTPUT)
summary_keep = [
    "target_variant",
    "panel_id",
    "lag_suffix",
    "family",
    "best_model",
    "validation_mae",
    "test_mae",
    "test_minus_validation_mae",
    "zero_change_baseline_mae",
    "delta_test_mae_vs_zero_change",
    "beats_zero_change_baseline",
    "test_spearman",
    "n_test",
]
comparison_keep = [
    "target_variant",
    "panel_id",
    "lag_suffix",
    "family",
    "best_model",
    "validation_mae",
    "test_mae",
    "test_minus_validation_mae",
    "zero_change_baseline_mae",
    "delta_test_mae_vs_zero_change",
    "test_spearman",
    "n_test",
]
if not persistence_summary.empty:
    delta_summary = persistence_summary[persistence_summary["target_variant"] == "delta_lag1"].copy()
    print("Selected best-family summary for delta_lag1")
    display(delta_summary.loc[:, [column for column in summary_keep if column in delta_summary]].round(4))

if not persistence_comparison.empty:
    delta_family_comparison = persistence_comparison[persistence_comparison["target_variant"] == "delta_lag1"].copy()
    print("Full family comparison slice for delta_lag1")
    display(
        delta_family_comparison.sort_values(["panel_id", "lag_suffix", "validation_mae"])
        .loc[:, [column for column in comparison_keep if column in delta_family_comparison]]
        .round(4)
    )

## Delta-To-Level Correction

Block-safe delta-to-level correction asks whether the predicted `delta_lag1` path improves the original level forecast. Each country is anchored at the latest pre-test target value, then predicted deltas are recursively added without using test-block target labels. Nonconsecutive anchor paths are flagged and excluded from corrected-level MAE, so one-year deltas are not used to bridge multi-year target gaps.

In [ ]:
persistence_level_correction = read_csv_or_empty(PERSISTENCE_LEVEL_CORRECTION_SUMMARY_OUTPUT)
persistence_level_predictions = read_csv_or_empty(PERSISTENCE_LEVEL_CORRECTION_OUTPUT)
if not persistence_level_correction.empty:
    correction_keep = [
        "panel_id",
        "lag_suffix",
        "family",
        "best_model",
        "n_test",
        "n_test_total",
        "n_test_excluded_anchor_gap",
        "global_fallback_anchor_rows",
        "history_only_mae",
        "corrected_level_mae",
        "delta_corrected_mae_minus_history",
        "beats_history_only",
        "corrected_level_spearman",
    ]
    selected_delta = pd.DataFrame()
    if not persistence_summary.empty:
        selected_delta = persistence_summary[persistence_summary["target_variant"] == "delta_lag1"][
            ["panel_id", "lag_suffix", "family", "best_model"]
        ].drop_duplicates()
    selected_level_correction = persistence_level_correction.merge(
        selected_delta,
        on=["panel_id", "lag_suffix", "family", "best_model"],
        how="inner",
    ) if not selected_delta.empty else persistence_level_correction.copy()
    print("Validation-selected block-safe delta-to-level correction")
    display(selected_level_correction.loc[:, [column for column in correction_keep if column in selected_level_correction]].round(4))
    print("Full block-safe delta-to-level correction family comparison")
    display(
        persistence_level_correction.sort_values(["panel_id", "lag_suffix", "family"])
        .loc[:, [column for column in correction_keep if column in persistence_level_correction]]
        .round(4)
    )

if not persistence_level_predictions.empty:
    audit_columns = [
        "target_variant",
        "panel_id",
        "lag_suffix",
        "family",
        "best_model",
        "forecast_path_eligible",
        "uses_test_label_for_anchor",
    ]
    audit = (
        persistence_level_predictions.loc[:, audit_columns]
        .assign(
            ineligible_path=lambda frame: ~frame["forecast_path_eligible"].astype(bool),
            test_label_anchor=lambda frame: frame["uses_test_label_for_anchor"].astype(bool),
        )
        .groupby(["target_variant", "panel_id", "lag_suffix", "family", "best_model"], as_index=False)
        .agg(
            forecast_rows=("forecast_path_eligible", "size"),
            ineligible_paths=("ineligible_path", "sum"),
            test_label_anchor_rows=("test_label_anchor", "sum"),
        )
    )
    display(audit.head(12))

## Delta Target Feature Roles

Feature roles are read from the validation-selected model within each delta panel and from native importance diagnostics across the delta candidate families. Tree importances are unsigned; linear coefficients retain direction only when the selected linear model has nonzero coefficients. Only rows with `normalized_importance > 0` are interpreted as active feature roles. Zero-shrunk coefficients are displayed separately when present and should not be counted as predictor roles.

In [ ]:
show_png(PERSISTENCE_ADJUSTED_FEATURE_FIGURE)
print("Artifact key: fig13_persistence_adjusted_delta_feature_roles")
persistence_top_features = read_csv_or_empty(PERSISTENCE_ADJUSTED_TOP_FEATURES_OUTPUT)
feature_keep = [
    "panel_id",
    "lag_suffix",
    "family",
    "model",
    "feature_rank",
    "feature_label",
    "importance_kind",
    "normalized_importance",
    "importance_direction",
    "test_mae",
    "zero_change_baseline_mae",
]
if not persistence_top_features.empty:
    delta_features = persistence_top_features[persistence_top_features["target_variant"] == "delta_lag1"].copy()
    active_delta_features = delta_features[delta_features["normalized_importance"] > 0].copy()
    zero_shrunk_delta_features = delta_features[delta_features["normalized_importance"] <= 0].copy()
    print("Active delta_lag1 feature roles from selected top-feature artifact")
    display(active_delta_features.loc[:, [column for column in feature_keep if column in active_delta_features]].round(4))
    if not zero_shrunk_delta_features.empty:
        print("Zero-shrunk coefficients not interpreted as active feature roles")
        display(
            zero_shrunk_delta_features.loc[
                :, [column for column in feature_keep if column in zero_shrunk_delta_features]
            ].round(4)
        )

persistence_native_importance = read_csv_or_empty(PERSISTENCE_ADJUSTED_NATIVE_IMPORTANCE_OUTPUT)
if not persistence_native_importance.empty:
    active_delta_native = persistence_native_importance[
        (persistence_native_importance["target_variant"] == "delta_lag1")
        & (persistence_native_importance["normalized_importance"] > 0)
    ].copy()
    native_keep = [
        "panel_id",
        "lag_suffix",
        "family",
        "model",
        "feature_label",
        "importance_kind",
        "signed_importance",
        "normalized_importance",
        "validation_mae",
        "test_mae",
    ]
    print("Active native-importance rows across delta_lag1 panels, lags, and model families")
    display(
        active_delta_native.sort_values(
            ["panel_id", "lag_suffix", "family", "normalized_importance"],
            ascending=[True, True, True, False],
        )
        .groupby(["panel_id", "lag_suffix", "family"], group_keys=False)
        .head(3)
        .loc[:, [column for column in native_keep if column in active_delta_native]]
        .round(4)
    )

## Log-Ratio Robustness

`log_ratio_lag1` is a robustness target rather than the main supplement. It checks sensitivity of the delta-target conclusion under a proportional-change target. Interpret it as target-scale sensitivity, not as a replacement for the delta task.

In [ ]:
if not persistence_summary.empty:
    ratio_summary = persistence_summary[persistence_summary["target_variant"] == "log_ratio_lag1"].copy()
    print("Selected best-family summary for log_ratio_lag1")
    display(ratio_summary.loc[:, [column for column in summary_keep if column in ratio_summary]].round(4))

if not persistence_comparison.empty:
    ratio_family_comparison = persistence_comparison[persistence_comparison["target_variant"] == "log_ratio_lag1"].copy()
    print("Full family comparison slice for log_ratio_lag1")
    display(
        ratio_family_comparison.sort_values(["panel_id", "lag_suffix", "validation_mae"])
        .loc[:, [column for column in comparison_keep if column in ratio_family_comparison]]
        .round(4)
    )

## Cross-Model Consistency

This section compares the top-ranked predictors across coefficient magnitude, tree impurity importance, and permutation importance. Treat agreement as descriptive stability across diagnostics, not as a formal discovery claim.

In [ ]:
top_lists = []
if not coefficients.empty:
    # Filter active predictors with abs_coefficient > 1e-12.
    nonzero_coefficients = coefficients[coefficients["abs_coefficient"] > 1e-12].copy()
    top_lists.append(
        nonzero_coefficients.sort_values("abs_coefficient", ascending=False)
        .loc[:, ["feature", "abs_coefficient"]]
        .head(8)
        .rename(columns={"abs_coefficient": "score"})
        .assign(diagnostic="linear_abs_coefficient")
    )
if not tree_importance.empty:
    top_lists.append(
        tree_importance.loc[:, ["feature", "importance"]]
        .head(8)
        .rename(columns={"importance": "score"})
        .assign(diagnostic="tree_impurity_importance")
    )
if not permutation_importance.empty:
    top_lists.append(
        permutation_importance.loc[:, ["feature", "permutation_importance_mean"]]
        .head(8)
        .rename(columns={"permutation_importance_mean": "score"})
        .assign(diagnostic="tree_permutation_importance")
    )

if top_lists:
    consistency = pd.concat(top_lists, ignore_index=True)
    display(consistency.loc[:, ["diagnostic", "feature", "score"]].round(4))
    display(
        consistency.groupby("feature", as_index=False)
        .agg(diagnostic_count=("diagnostic", "nunique"))
        .sort_values(["diagnostic_count", "feature"], ascending=[False, True])
    )


## Error-Aware Interpretation

Interpretability should be read together with failure modes. If a predictor appears important but errors concentrate in specific years, countries, or target ranges, the explanation is conditional on those weaknesses.

In [ ]:
show_indexed_figure(FIGURE_INDEX_OUTPUT, "top_absolute_errors", FIGURES_DIR / "linear_model_top_absolute_errors.png")
top_errors = read_csv_or_empty(TOP_ERRORS_OUTPUT)
if not top_errors.empty:
    keep = [column for column in ["country_name", "year", "observed", "prediction", "absolute_error"] if column in top_errors]
    display(top_errors.loc[:, keep].head(12).round(4))

show_indexed_figure(FIGURE_INDEX_OUTPUT, "error_by_year", FIGURES_DIR / "linear_model_error_by_year.png")
error_by_year = read_csv_or_empty(ERROR_BY_YEAR_OUTPUT)
if not error_by_year.empty:
    display(error_by_year.round(4))

error_by_quantile = read_csv_or_empty(ERROR_BY_TARGET_QUANTILE_OUTPUT)
if not error_by_quantile.empty:
    display(error_by_quantile.round(4))


## Reviewer Caveats

- These diagnostics explain fitted prediction behavior, not causal policy effects.
- Tree impurity importance can favor continuous or high-variance predictors; permutation importance and SHAP are included as robustness checks.
- PDP curves can be off-manifold in correlated country-panel data, so read them as qualitative response diagnostics.
- Cross-model agreement is descriptive stability, not variable-selection proof.
- Delta/log-ratio diagnostics answer change-target questions and should not be mixed with level-target importance claims.
- Delta-to-level correction is a block-safe level forecast diagnostic; rows with nonconsecutive pre-test anchors are excluded from corrected-level MAE and reported separately.
- Any explanatory claim should be bounded by the persistence result: external predictors do not reliably beat national historical persistence on the latest test block.